In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')

# 1. join relasi

In [3]:
# guru_df = relasi_guru_mapel_df = mapel-df
join = (
    relasi_guru_mapel_df
    .merge(guru_df, on="guru_id", how="left")
    .merge(mapel_df, on="mapel_id", how="left")
)

# penambahan kolom "total"
join["total"] = (
    join.groupby("guru_id")["durasi"]
    .transform("sum")
) 
join.head(3)

,guru_id,mapel_id,tingkatan,durasi,nama_guru,kode_mapel,nama_mapel,MGMP,jam_per_minggu,total
0,2,6,7,24,"MULYANI, S.Pd., M.Pd.",F,ILMU PENGETAHUAN SOSIAL,Rabu,3,24
1,3,6,7,12,"SUDARYANTO, S.Pd.",F,ILMU PENGETAHUAN SOSIAL,Rabu,3,24
2,3,6,9,12,"SUDARYANTO, S.Pd.",F,ILMU PENGETAHUAN SOSIAL,Rabu,3,24


In [4]:
join.count()

guru_id           93
mapel_id          93
tingkatan         93
durasi            93
nama_guru         93
kode_mapel        93
nama_mapel        93
MGMP              93
jam_per_minggu    93
total             93
dtype: int64

# 2. Menggunakan Numerik

### konversi hari ke numerik

In [5]:
# dictionary hari -> numerik
mapping_hari = {
    "Senin": 1,
    "Selasa": 2,
    "Rabu": 3,
    "Kamis": 4,
    "Jumat": 5,
}

### mengambil kolom penting

In [6]:
# mengambil kolom numerik
relasi = join[[
    "guru_id",
    "mapel_id",
    "jam_per_minggu",
    "tingkatan",
    "durasi",
    "total",
    "MGMP"
]].copy()

# melakukan konversi hari ke numerik pakai mapping
relasi["MGMP"] = relasi["MGMP"].map(mapping_hari)
relasi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   guru_id         93 non-null     int64
 1   mapel_id        93 non-null     int64
 2   jam_per_minggu  93 non-null     int64
 3   tingkatan       93 non-null     int64
 4   durasi          93 non-null     int64
 5   total           93 non-null     int64
 6   MGMP            93 non-null     int64
dtypes: int64(7)
memory usage: 5.2 KB


### mapping hari dan jumlah slot

In [7]:
# mapping jumlah slot per hari
slot_per_hari = {
    1 : 8, # senin
    2 : 8, # Selasa
    3 : 8, # rabu
    4 : 7, # kamis
    5 : 5, # jumat
}

### count kelas

In [8]:
# mengambil total kelas (27)
total_kelas = kelas_df["kelas_id"].count()
total_kelas

27

# 3. Groupping guru  
mapel_id dan tingkatan

In [9]:
guru_by_mapel = (
    relasi
    .groupby(["mapel_id", "tingkatan"])["guru_id"]
    .apply(list)
    .to_dict()
)
guru_by_mapel

{(1, 7): [31, 39],
 (1, 8): [24, 31],
 (1, 9): [26],
 (2, 7): [33, 39, 45],
 (2, 8): [30, 33, 41],
 (2, 9): [10, 34],
 (3, 7): [22, 29, 36],
 (3, 8): [40, 45],
 (3, 9): [6, 20, 22],
 (4, 7): [19, 47],
 (4, 8): [14, 28],
 (4, 9): [11, 12, 28],
 (5, 7): [18, 27, 48],
 (5, 8): [13, 46, 48],
 (5, 9): [8, 27],
 (6, 7): [2, 3, 44],
 (6, 8): [24, 25, 41, 44],
 (6, 9): [3, 9],
 (7, 7): [21, 37],
 (7, 8): [34, 37],
 (7, 9): [4, 7, 25],
 (8, 7): [5, 17, 23, 42],
 (8, 8): [5, 17, 23, 42],
 (8, 9): [5, 17, 23],
 (9, 7): [38, 42],
 (9, 8): [35, 38, 42],
 (9, 9): [35],
 (10, 7): [15, 16, 28, 31, 46, 47, 48],
 (10, 8): [16],
 (10, 9): [15],
 (11, 7): [37, 43],
 (11, 8): [32, 40],
 (11, 9): [32],
 (12, 7): [51, 55],
 (12, 8): [50, 54],
 (12, 9): [49, 54],
 (13, 7): [53],
 (13, 8): [52, 53],
 (13, 9): [52]}

# 4. Daftar Variabel

variabel variabel yang telah diproses dan siap digunakan
| Nama Variabel| Keterangan Singkat |
|--------------|-------------------|
| `guru_df` |  Original guru |
| `kelas_df` |  Original kelas |
| `mapel_df` |  Original mapel |
| `relasi_guru_mapel_df` |  Original relasi |
| `slot_df` |  Original slot |
| `relasi` |  Relasi guru & mapel (93 rows) |
| `guru_by_mapel` |  Guru groupping |
| `total_kelas`| 27 (int)|
|`slot_per_hari`| dictionary |


# 5. Fungsi Inisialisasi Individu

In [44]:
def individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi):

    mapel_ids = list(range(1, 14))     # 13 mapel
    tingkatan = relasi["tingkatan"].iloc[0]

    # preprocess guru per mapel
    guru_mapel_list = {
        m: guru_by_mapel.get((m, tingkatan), [])
        for m in mapel_ids
    }

    individu = []

    for _ in range(total_kelas):

        kelas = []

        # ===== GEN MAPEL PER HARI =====
        for hari in sorted(slot_per_hari.keys()):  # 1..5
            jumlah_slot = slot_per_hari[hari]
            gen_hari = [random.choice(mapel_ids) for _ in range(jumlah_slot)]
            kelas.append(gen_hari)

        # ===== GEN GURU (13 MAPEL) =====
        gen_guru = [
            random.choice(guru_mapel_list[m]) if guru_mapel_list[m] else 0
            for m in mapel_ids
        ]

        kelas.append(gen_guru)

        individu.append(kelas)

    return individu


In [45]:
individu = individuConstruct(slot_per_hari, total_kelas, guru_by_mapel, relasi)
individu

[[[11, 8, 5, 7, 12, 2, 4, 4],
  [7, 4, 5, 8, 6, 9, 9, 11],
  [11, 3, 5, 7, 8, 4, 11, 2],
  [5, 11, 12, 9, 8, 12, 10],
  [2, 5, 6, 5, 3],
  [39, 45, 29, 19, 18, 2, 21, 23, 42, 15, 37, 51, 53]],
 [[6, 3, 7, 13, 5, 7, 10, 1],
  [12, 3, 11, 6, 7, 13, 10, 11],
  [6, 3, 5, 3, 4, 6, 3, 8],
  [13, 3, 9, 11, 1, 6, 5],
  [3, 7, 2, 5, 13],
  [31, 45, 29, 47, 27, 3, 21, 17, 38, 16, 43, 51, 53]],
 [[12, 6, 1, 6, 5, 7, 8, 6],
  [1, 2, 11, 1, 9, 13, 5, 7],
  [12, 8, 1, 10, 11, 4, 11, 11],
  [8, 13, 13, 7, 8, 8, 4],
  [12, 11, 2, 13, 10],
  [39, 45, 29, 47, 18, 2, 37, 23, 42, 16, 43, 51, 53]],
 [[8, 2, 9, 7, 13, 9, 1, 5],
  [9, 2, 9, 12, 1, 5, 11, 10],
  [3, 10, 5, 13, 4, 2, 6, 5],
  [1, 5, 12, 4, 12, 8, 8],
  [1, 6, 3, 4, 10],
  [39, 33, 22, 47, 48, 3, 21, 23, 38, 28, 37, 51, 53]],
 [[2, 13, 5, 3, 9, 1, 12, 3],
  [3, 1, 13, 7, 2, 7, 12, 12],
  [1, 3, 8, 11, 8, 3, 12, 1],
  [7, 7, 5, 8, 4, 7, 13],
  [7, 6, 3, 8, 11],
  [31, 45, 29, 47, 27, 3, 21, 17, 38, 47, 43, 51, 53]],
 [[3, 7, 4, 12, 2, 2, 5, 7],


individu = 
[ # ini 1 individu
        [ # ini 1 kelas
                [m1, m2, m3, m4, m5, m6, m7, m8],   # Senin (8)
                [m1, m2, m3, m4, m5, m6, m7, m8],   # Selasa (8)
                [m1, m2, m3, m4, m5, m6, m7, m8],   # Rabu (8)
                [m1, m2, m3, m4, m5, m6, m7],       # Kamis (7)
                [m1, m2, m3, m4, m5],                # Jumat (5)
                [g1, g2, g3, g4, g5, sampai g13]        # guru pengajar (13)
        ],
        [
                # kelas lain
        ]
]
        
    